# Experiment B: Custom YOLOv8n Fine-Tuning on Gate Dataset
### AI-Based Smart Gate Monitoring System
**Author:** Siddharth Dhanush R (Team Lead)
**Objective:** Fine-tune pretrained `yolov8n.pt` on the annotated campus gate dataset for 50 epochs using Google Colab GPU acceleration.

## 1. Verify Google Colab GPU & Install Ultralytics

In [ ]:
!pip install -q ultralytics

import torch
import os
from ultralytics import YOLO

assert torch.cuda.is_available(), "[ERROR] GPU is not active. Navigate to Runtime -> Change runtime type -> T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"[INFO] Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")

# Determine recommended batch size based on available VRAM
batch_size = 16 if vram_gb >= 11.0 else 8
print(f"[INFO] Configured Batch Size: {batch_size}")

## 2. Mount Google Drive & Locate Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Path to data.yaml on Google Drive or unpacked in /content
DATA_YAML_PATH = '/content/drive/MyDrive/GateDataset/data.yaml'
if not os.path.exists(DATA_YAML_PATH):
    print(f"[WARNING] {DATA_YAML_PATH} not found. Please verify dataset location.")
else:
    print(f"[INFO] Found dataset configuration at: {DATA_YAML_PATH}")

## 3. Initialize Model with Pretrained YOLOv8n Weights

In [ ]:
# Load pretrained weights - transfer learning starting point
model = YOLO('yolov8n.pt')
print("[INFO] Loaded pretrained yolov8n.pt weights for fine-tuning.")

## 4. Execute Fine-Tuning (50 Epochs)

In [ ]:
# Execute training run
results = model.train(
    data=DATA_YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=batch_size,
    patience=10,         # Early stopping if no mAP improvement
    save=True,
    project='gate_monitoring_runs',
    name='yolov8n_custom_gate',
    device=0,
    workers=2,
    verbose=True
)
print("[INFO] Training finished successfully.")

## 5. Preserve & Copy `best.pt` Checkpoint

In [ ]:
import shutil

best_weights = 'gate_monitoring_runs/yolov8n_custom_gate/weights/best.pt'
target_drive_path = '/content/drive/MyDrive/GateMonitoringModels/gate_yolov8n_best.pt'
os.makedirs(os.path.dirname(target_drive_path), exist_ok=True)

if os.path.exists(best_weights):
    shutil.copy(best_weights, target_drive_path)
    print(f"[SUCCESS] Checkpoint preserved to Google Drive at: {target_drive_path}")
else:
    print(f"[ERROR] Checkpoint not found at {best_weights}")